In [ ]:
# ==========================================
# SMOGNET COMPLETE ML PIPELINE
# ==========================================

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import warnings
import os # Added for path manipulation

warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: LOAD TRAINING FILES
# ==========================================

train_files=[

"concatenated_dataset_Aug_2021_to_July_2024.csv",
"islamabad_complete_data.xlsx",
"karachi_complete_data.xlsx",
"lahore_complete_data.xlsx",
"peshawar_complete_data.csv",
"quetta_complete_data.csv"

]

train_data=[]

for file in train_files:

    print("Loading:",file)

    if file.endswith(".xlsx"):
        temp=pd.read_excel(file)

    elif file.endswith(".csv"):
        temp=pd.read_csv(file)

    else:
        print(f"Warning: Unhandled file type for {file}. Attempting to read as excel.")
        temp=pd.read_excel(file)

    # Extract city name and add as a 'City' column
    base_name = os.path.basename(file)
    if "_complete_data" in base_name: # This covers most city-specific files
        city_name = base_name.split('_')[0].capitalize()
        temp["City"] = city_name
    elif "concatenated_dataset" in base_name: # Specific handling for the concatenated file
        temp["City"] = "Aggregated" # Assign a placeholder for the aggregated dataset
    else: # Fallback for any other unexpected filenames
        temp["City"] = "Unknown"

    train_data.append(temp)

train_df=pd.concat(
    train_data,
    ignore_index=True
)

print("\nTraining shape:")
print(train_df.shape)

# ==========================================
# STEP 2: LOAD TEST FILES
# ==========================================

test_files=[

"islamabad_complete_data_july_to_dec_2024.csv",
"karachi_complete_data_july_to_dec_2024.csv",
"lahore_complete_data_july_to_dec_2024.csv",
"peshawar_complete_data_july_to_dec_2024.csv",
"quetta_complete_data_july_to_dec_2024.csv"

]

test_data=[]

for file in test_files:

    print("Loading:",file)

    if file.endswith(".xlsx"):
        temp=pd.read_excel(file)

    elif file.endswith(".csv"):
        temp=pd.read_csv(file)

    else:
        print(f"Warning: Unhandled file type for {file}. Attempting to read as excel.")
        temp=pd.read_excel(file)

    # All test files are city-specific based on the naming convention
    base_name = os.path.basename(file)
    city_name = base_name.split('_')[0].capitalize()
    temp["City"] = city_name

    test_data.append(temp)

test_df=pd.concat(
    test_data,
    ignore_index=True
)

print("\nTesting shape:")
print(test_df.shape)

# ==========================================
# STEP 3: CLEAN DATA
# ==========================================

train_df.fillna(
    method="ffill",
    inplace=True
)

test_df.fillna(
    method="ffill",
    inplace=True
)

# ==========================================
# IMPORTANT
# Match names with your dataset columns
# ==========================================

features=[

'components_pm2_5',
'components_pm10',
'components_no',
'components_no2',
'components_so2',
'components_nh3',
'components_co',
'components_o3'

]

X_train=train_df[features]
X_test=test_df[features]

# ==========================================
# STAGE 1
# ANOMALY DETECTION
# ==========================================

print("\nTraining anomaly model...")

anomaly_model=IsolationForest(

    contamination=0.05,
    n_estimators=200,
    random_state=42

)

anomaly_model.fit(
    X_train
)

test_df["Anomaly"]=(
    anomaly_model.predict(X_test)
)

test_df["Anomaly"]=(
    test_df["Anomaly"]
    .map({1:"Normal",-1:"Spike"})
)

print("\nAnomaly Results")

print(
test_df["Anomaly"].value_counts()
)

# ==========================================
# STAGE 2
# CREATE LABELS
# ==========================================

def classify_source(row):

    if row['components_nh3']>80 and row['components_co']>20:
        return "Crop Burning"

    elif row['components_no']>50 and row['components_no2']>50:
        return "Vehicular"

    elif row['components_so2']>40:
        return "Industrial"

    elif row['components_pm10']>row['components_pm2_5']*2:
        return "Dust Storm"

    else:
        return "Mixed"

train_df["Source"]=(
train_df.apply(
classify_source,
axis=1
)
)

encoder=LabelEncoder()

y=encoder.fit_transform(
train_df["Source"]
)

# ==========================================
# STAGE 3
# SOURCE CLASSIFICATION
# ==========================================

print("\nTraining classifier...")

classifier=RandomForestClassifier(

n_estimators=300,
max_depth=10,
random_state=42

)

classifier.fit(
X_train,
y
)

pred=classifier.predict(
X_test
)

test_df["Predicted_Source"]=(
encoder.inverse_transform(pred)
)

# ==========================================
# STAGE 4
# ALERT GENERATION
# ==========================================

def generate_alert(row):

    city=row["City"]

    source=row["Predicted_Source"]

    status=row["Anomaly"]

    if status=="Spike":

        return f"""
Air quality in {city} has reached unhealthy levels.
Likely pollution source is {source}.
Children, elderly people and respiratory patients should avoid outdoor activity.
Wear masks and reduce exposure.
"""

    else:

        return "Air quality normal."


test_df["Alert"]=(
test_df.apply(
generate_alert,
axis=1
)
)

# ==========================================
# FINAL RESULTS
# ==========================================

print("\nSample Output")

print(

test_df[
[
'City',
'Anomaly',
'Predicted_Source',
'Alert'
]
].head()

)

# ==========================================
# SAVE OUTPUT
# ==========================================

test_df.to_csv(

"SmogNet_Final_Output.csv",
index=False

)

print("\nSaved Successfully")
print("Output File: SmogNet_Final_Output.csv")

Loading: concatenated_dataset_Aug_2021_to_July_2024.csv
Loading: islamabad_complete_data.xlsx
Loading: karachi_complete_data.xlsx
Loading: lahore_complete_data.xlsx
Loading: peshawar_complete_data.csv
Loading: quetta_complete_data.csv

Training shape:
(264335, 28)
Loading: islamabad_complete_data_july_to_dec_2024.csv
Loading: karachi_complete_data_july_to_dec_2024.csv
Loading: lahore_complete_data_july_to_dec_2024.csv
Loading: peshawar_complete_data_july_to_dec_2024.csv
Loading: quetta_complete_data_july_to_dec_2024.csv

Testing shape:
(21792, 19)

Training anomaly model...

Anomaly Results
Anomaly
Normal    19364
Spike      2428
Name: count, dtype: int64

Training classifier...

Sample Output
        City Anomaly Predicted_Source                Alert
0  Islamabad  Normal            Mixed  Air quality normal.
1  Islamabad  Normal            Mixed  Air quality normal.
2  Islamabad  Normal            Mixed  Air quality normal.
3  Islamabad  Normal            Mixed  Air quality normal.
4 

In [ ]:
!pip install xgboost
!pip install scikit-learn
!pip install pandas
!pip install openpyxl

In [ ]:
# ==========================================
# MULTI MODEL COMPARISON
# BEST MODEL AUTO SELECTION
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from sklearn.ensemble import (
RandomForestClassifier,
GradientBoostingClassifier,
ExtraTreesClassifier,
AdaBoostClassifier
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

# ==========================================
# Train/Test Split
# ==========================================

X = train_df[features]

y = encoder.fit_transform(
    train_df["Source"]
)

X_train, X_valid, y_train, y_valid = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)

# ==========================================
# MODELS
# ==========================================

models={

"Random Forest":
RandomForestClassifier(
n_estimators=300,
random_state=42
),

"Gradient Boost":
GradientBoostingClassifier(),

"Extra Trees":
ExtraTreesClassifier(
n_estimators=300
),

"AdaBoost":
AdaBoostClassifier(),

"Decision Tree":
DecisionTreeClassifier(),

"Logistic Regression":
LogisticRegression(
max_iter=1000
),

"KNN":
KNeighborsClassifier(),

"XGBoost":
XGBClassifier(
n_estimators=300,
learning_rate=0.1,
max_depth=6
)

}

# ==========================================
# TRAIN + COMPARE
# ==========================================

best_accuracy=0
best_model=None
best_model_name=""

results=[]

print("\nTraining Models...\n")

for name,model in models.items():

    print("Training:",name)

    model.fit(
        X_train,
        y_train
    )

    pred=model.predict(
        X_valid
    )

    acc=accuracy_score(
        y_valid,
        pred
    )

    results.append([name,acc])

    print(
    f"{name}: {acc:.4f}"
    )

    if acc>best_accuracy:

        best_accuracy=acc
        best_model=model
        best_model_name=name


# ==========================================
# RESULTS TABLE
# ==========================================

results_df=pd.DataFrame(

results,
columns=[
"Model",
"Accuracy"
]

)

results_df=results_df.sort_values(
"Accuracy",
ascending=False
)

print("\n====================")
print("MODEL RANKING")
print("====================")

print(results_df)

print("\nBEST MODEL:")
print(best_model_name)

print(
"Accuracy:",
best_accuracy
)

# ==========================================
# FINAL PREDICTION USING BEST MODEL
# ==========================================

print("\nRunning final predictions...")

final_prediction=best_model.predict(
X_test
)

test_df["Predicted_Source"]=(
encoder.inverse_transform(
final_prediction
)
)

# ==========================================
# Detailed report
# ==========================================

pred=best_model.predict(
X_valid
)

print("\nClassification Report")

print(

classification_report(
y_valid,
pred,
target_names=encoder.classes_
)

)

# ==========================================
# Save best output
# ==========================================

test_df.to_csv(

"SmogNet_BestModel_Output.csv",
index=False

)

print(
"\nSaved: SmogNet_BestModel_Output.csv"
)


Training Models...

Training: Random Forest
Random Forest: 0.9946
Training: Gradient Boost
Gradient Boost: 0.9930
Training: Extra Trees
Extra Trees: 0.9900
Training: AdaBoost
AdaBoost: 0.9387
Training: Decision Tree
Decision Tree: 0.9925
Training: Logistic Regression
Logistic Regression: 0.9557
Training: KNN
KNN: 0.9622
Training: XGBoost
XGBoost: 0.9953

MODEL RANKING
                 Model  Accuracy
7              XGBoost  0.995328
0        Random Forest  0.994571
1       Gradient Boost  0.993020
4        Decision Tree  0.992472
2          Extra Trees  0.989975
6                  KNN  0.962245
5  Logistic Regression  0.955719
3             AdaBoost  0.938676

BEST MODEL:
XGBoost
Accuracy: 0.9953278983108556

Running final predictions...

Classification Report
              precision    recall  f1-score   support

Crop Burning       0.94      0.95      0.95       991
  Dust Storm       1.00      1.00      1.00     34535
  Industrial       0.99      1.00      1.00       693
       Mixe

In [ ]:
# ==========================================
# SAVE MODEL
# ==========================================

import joblib

# Save best model
joblib.dump(
    best_model,
    "best_smognet_model.pkl"
)

# Save label encoder
joblib.dump(
    encoder,
    "label_encoder.pkl"
)

print("\nModel Saved Successfully")
print("Saved: best_smognet_model.pkl")
print("Saved: label_encoder.pkl")


# ==========================================
# LOAD MODEL AGAIN
# ==========================================

loaded_model = joblib.load(
    "best_smognet_model.pkl"
)

loaded_encoder = joblib.load(
    "label_encoder.pkl"
)

print("\nModel Loaded Successfully")


# ==========================================_
# NEW PREDICTION
# ==========================================

new_data = pd.DataFrame({

'components_pm2_5':[250],
'components_pm10':[420],
'components_no':[75],
'components_no2':[90],
'components_so2':[15],
'components_nh3':[85],
'components_co':[30],
'components_o3':[20]

})


prediction = loaded_model.predict(
    new_data
)

predicted_source = (
loaded_encoder.inverse_transform(
prediction
)
)

print("\nPrediction Result")
print(
"Predicted Source:",
predicted_source[0]
)


# ==========================================
# ANOMALY PREDICTION
# ==========================================

anomaly_prediction = anomaly_model.predict(
    new_data
)

if anomaly_prediction[0]==-1:

    anomaly_status="Spike"

else:

    anomaly_status="Normal"

print(
"Anomaly:",
anomaly_status
)


# ==========================================
# GENERATE ALERT
# ==========================================

if anomaly_status=="Spike":

    alert=f"""
Air quality has reached unhealthy levels.
Likely source: {predicted_source[0]}
Children, elderly people and respiratory patients should avoid outdoor activity.
Wear masks and reduce exposure.
"""

else:

    alert="Air quality normal"

print("\nGenerated Alert:")
print(alert)


Model Saved Successfully
Saved: best_smognet_model.pkl
Saved: label_encoder.pkl

Model Loaded Successfully

Prediction Result
Predicted Source: Dust Storm
Anomaly: Spike

Generated Alert:

Air quality has reached unhealthy levels.
Likely source: Dust Storm
Children, elderly people and respiratory patients should avoid outdoor activity.
Wear masks and reduce exposure.



In [ ]:
# ==================================
# SAVE PREDICTION RESULTS
# ==================================

prediction_result = pd.DataFrame({

    "PM2.5":[250],
    "PM10":[420],
    "NO":[75],
    "NO2":[90],
    "SO2":[15],
    "NH3":[85],
    "CO":[30],
    "O3":[20],

    "Predicted_Source":[predicted_source[0]],
    "Anomaly":[anomaly_status],
    "Alert":[alert]

})

prediction_result.to_csv(
    "Live_Prediction_Output.csv",
    index=False
)

print("\nPrediction file saved")
print("Saved: Live_Prediction_Output.csv")


Prediction file saved
Saved: Live_Prediction_Output.csv


In [ ]:
# Prediction confidence

probabilities = loaded_model.predict_proba(new_data)

confidence = probabilities.max()*100

print(
f"Prediction Confidence: {confidence:.2f}%"
)

Prediction Confidence: 82.43%
